In [6]:
from itertools import chain
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    GPT2Config,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
)

In [7]:
dataset = load_dataset("bookcorpus", trust_remote_code=True)
dataset = dataset["train"].train_test_split(test_size=0.0015) 

In [8]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [9]:
def tokenizer_function(example):
    return tokenizer(text=example["text"])

tokenized_ds = dataset.map(tokenizer_function, batched=True, batch_size=1000, num_proc=8, remove_columns="text")

Map (num_proc=8):   0%|          | 0/73893221 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (9824 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1495 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2047 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1126 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (9249 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence leng

Map (num_proc=8):   0%|          | 0/111007 [00:00<?, ? examples/s]

In [ ]:
def concat(examples):
    examples["input_ids"] = [list(chain.from_iterable(examples["input_ids"]))]

concated_ds = tokenized_ds.map(concat, batched=True, batch_size=1_000_000, num_proc=8)

Map (num_proc=8):   0%|          | 0/73893221 [00:00<?, ? examples/s]

TypeError: 'float' object cannot be interpreted as an integer

In [ ]:
def chunk(examples, chunk_size=1024):
    input_ids = examples["input_ids"][0]
    attention_mask = examples["attention_mask"][0]
    input_ids_truncated = []
    attention_mask_truncated = []
    # slice with step_size = chunk_size
    for i in range(0, len(input_ids), chunk_size):
        chunk = input_ids[i:i+chunk_size]
        # drop the last chunk if not equal
        if len(chunk) == chunk_size:
            input_ids_truncated.append(chunk)
            attention_mask_truncated.append(attention_mask[i:i+chunk_size])
    examples["input_ids"] = input_ids_truncated
    examples["attention_mask"] = attention_mask_truncated
    return examples

chunked_ds = concated_ds.map(chunk, batched=True, batch_size=2, num_proc=2)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
print(f"Train size: {len(chunked_ds['train'])}")
print(f"Eval size: {len(chunked_ds['test'])}")

In [ ]:
configuration = GPT2Config()
model = GPT2LMHeadModel(configuration)

In [ ]:
training_args = TrainingArguments(
    output_dir="outputs/gpt2",
    eval_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    adam_beta1=0.9,
    adam_beta2=0.999,                                  
    weight_decay=0.01,                                  
    save_steps=5000,
    save_total_limit=10,                                  
    gradient_checkpointing=True,
) 
trainer = Trainer(
    model=model,
    args = training_args,
    tokenizer=tokenizer,
    train_dataset=chunked_ds["train"],
    eval_dataset=chunked_ds["test"],
    data_collator = data_collator
)
trainer.train()

In [ ]:
model = GPT2LMHeadModel.from_pretrained("outputs/gpt2/checkpoint-10/")
prompts = "I was telling her that"
inputs = tokenizer(prompts,return_tensors="pt").input_ids
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=10, top_p=0.95)
tokenizer.batch_decode(outputs, skip_special_tokens=True)